# Problem Set 11: Project Genesis - The Scholar-Prime (Week 11)

This notebook contains the Week 11 solution for **Scholar-Prime**, an academic research agent that uses Google DeepMind Science-Skills to query scientific databases and extract material parameters for simulation modeling.

## Objectives
1. Set up Science-Skills and verify the OpenAlex CLI.
2. Define the ADK agent in `cognitive_core/agent.py`.
3. Bind an arXiv search wrapper as an ADK tool.
4. Run a parameter extraction pipeline and write `docs/simulation_parameters.json`.

The external `science-skills/` checkout is intentionally ignored by Git. Clone it locally into this Set11 folder before running the live CLI cells.

## Exercise 1: Setting up the Science Skills

Clone the official repository into this folder if it is not already present:

```bash
git clone https://github.com/google-deepmind/science-skills.git
```

The current repository uses underscore folder names such as `literature_search_openalex` and `literature_search_arxiv`; the assignment text may show hyphenated names. The validation cell below supports both variants.

Official verification command:

```bash
uv run scripts/openalex_cli.py resolve authors "Geoffrey Hinton"
```

The console output is also saved to `docs/openalex_author_resolution.txt`.

In [1]:
import subprocess
from pathlib import Path

SET_ROOT = Path.cwd()
SCIENCE_SKILLS = SET_ROOT / "science-skills"

def find_skill_dir(*names: str) -> Path:
    for name in names:
        candidate = SCIENCE_SKILLS / "skills" / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Missing Science-Skills folder. Checked: {names}")

openalex_dir = find_skill_dir("literature_search_openalex", "literature-search-openalex")
cmd = ["uv", "run", "scripts/openalex_cli.py", "resolve", "authors", "Geoffrey Hinton"]
res = subprocess.run(cmd, cwd=openalex_dir, capture_output=True, text=True, timeout=90, check=False)

output = res.stdout if res.returncode == 0 else res.stderr
Path("docs").mkdir(exist_ok=True)
Path("docs/openalex_author_resolution.txt").write_text(output, encoding="utf-8")

print(output)
assert res.returncode == 0, output
assert "openalex.org" in output.lower()

[
  {
    "id": "https://openalex.org/A5108093963",
    "display_name": "Geoffrey E. Hinton",
    "hint": 385
  },
  {
    "id": "https://openalex.org/A5110248343",
    "display_name": "Geoffrey E. Hinton",
    "hint": 36
  },
  {
    "id": "https://openalex.org/A5000300454",
    "display_name": "Saurabh Saxena",
    "hint": 63
  },
  {
    "id": "https://openalex.org/A5002428732",
    "display_name": "Geoffrey F. Hinton",
    "hint": 2
  },
  {
    "id": "https://openalex.org/A5098035523",
    "display_name": "James A. Anderson and Geoffrey E. Hinton",
    "hint": 1
  }
]



## Exercise 2: Building the Literature Retrieval Agent

The ADK agent is implemented in `cognitive_core/agent.py`. It exposes `scholar_prime` as `root_agent` for the ADK Web UI and binds the literature-search and extraction tools.

The model defaults to `gemini-3.5-flash` and can be overridden with `SCHOLAR_PRIME_MODEL`.

In [2]:
from pathlib import Path

agent_code = Path("cognitive_core/agent.py").read_text(encoding="utf-8")
print(agent_code)

from __future__ import annotations

import os

from google.adk.agents.llm_agent import Agent

from .tools import extract_parameters_from_text, search_arxiv


scholar_prime = Agent(
    model=os.getenv("SCHOLAR_PRIME_MODEL", "gemini-3.5-flash"),
    name="scholar_prime",
    description=(
        "An academic research agent specialized in querying scientific "
        "databases and extracting material parameters."
    ),
    instruction=(
        "You are Scholar-Prime, a precise academic research agent for "
        "simulation modeling. Search scientific literature with the available "
        "tools, compare paper relevance from titles and abstracts, extract "
        "material parameters and formulas only when supported by the source "
        "text, and always report DOI or arXiv IDs when available. If a DOI is "
        "missing, state that explicitly instead of inventing one."
    ),
    tools=[search_arxiv, extract_parameters_from_text],
)

root_agent = scholar_prime




## Exercise 3: Automated Search & Downloader (Tool Binding)

`search_arxiv(query: str, max_results: int = 5) -> str` is implemented in `cognitive_core/tools.py` and bound in the ADK agent's `tools` list.

ADK Web UI test prompt:

```text
Scholar-Prime, search arXiv for papers on 'thermodynamic simulation parameters for advanced fission reactors'. Identify the most relevant paper and summarize its abstract.
```

If the live arXiv endpoint is slow or unavailable, the wrapper returns a structured timeout error rather than blocking indefinitely.

In [3]:
from cognitive_core.tools import search_arxiv

# Live test. This requires network access to export.arxiv.org and may time out
# in restricted environments. Uncomment for the final Web UI validation.
# print(search_arxiv("thermodynamic simulation parameters for advanced fission reactors", max_results=3))

print("search_arxiv is implemented and bound in cognitive_core/agent.py")

search_arxiv is implemented and bound in cognitive_core/agent.py


## Exercise 4: Parameter Extraction & Verification

`extract_parameters_from_text(text: str) -> dict` is implemented in `cognitive_core/tools.py`.

The pipeline script `scripts/extract_simulation_parameters.py` searches the literature, selects the top abstract, extracts parameters, and writes `docs/simulation_parameters.json`. The `--mock` flag provides deterministic local validation when the live arXiv endpoint is unavailable.

In [4]:
import json
from pathlib import Path

from scripts.extract_simulation_parameters import run_pipeline

result = run_pipeline(
    query="thermodynamic simulation parameters for advanced fission reactors",
    max_results=5,
    output_path=Path("docs/simulation_parameters.json"),
    use_mock=True,
)

print(json.dumps(result, indent=2))
assert Path("docs/simulation_parameters.json").exists()
assert result["extraction"]["parameters"]

{
  "status": "success",
  "query": "thermodynamic simulation parameters for advanced fission reactors",
  "agent_name": "scholar_prime",
  "source_paper": {
    "title": "Mock validation excerpt for UO2 material parameters",
    "authors": [],
    "published": null,
    "arxiv_id": null,
    "doi": "10.1016/j.jnucmat.2019.01.001",
    "pdf_url": null
  },
  "extraction": {
    "material": "Uranium Dioxide",
    "doi": "10.1016/j.jnucmat.2019.01.001",
    "parameters": [
      {
        "parameter": "thermal_conductivity",
        "value": 3.5,
        "unit": "W/(m*K)",
        "evidence": "reported density is 10.97 g/cm3, the melting point is 3120 K, and the baseline thermal conductivity is 3.5 W/(m*K). Reference DOI: 10.1016/j.jnucmat.2019.01.001.",
        "confidence": "medium"
      },
      {
        "parameter": "melting_point",
        "value": 3120.0,
        "unit": "K",
        "evidence": "m Dioxide (UO2) reactor fuel material. The reported density is 10.97 g/cm3, the melt